In [1]:
from src.data_loader import PROCESSED_DATA_DIR, MODELS_DIR
from features import MarketDemandModel_data_transformation, augment_price_elasticity
import config
import pandas as pd
import numpy as np

import joblib
import optuna


import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

C:\Users\UserB\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet(PROCESSED_DATA_DIR / 'train.parquet')

In [3]:
X , y = MarketDemandModel_data_transformation(df, tick_size = 7)

In [4]:
X_train, y_train = augment_price_elasticity(X, y)

C:\Users\UserB\Documents\ML\Online Retail II UCI\src\features.py:129: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  X_high['Price_vs_Avg'] = X_high.groupby('StockCode')['Price'].transform(
C:\Users\UserB\Documents\ML\Online Retail II UCI\src\features.py:133: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  X_high['Price_fall_flag'] = (X_high['Price'] < X_high.groupby('StockCode')['Price'].shift(1)).astype(int)
C:\Users\UserB\Documents\ML\Online Retail II UCI\src\features.py:134: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fal

In [5]:
tcsv = TimeSeriesSplit(n_splits=5)
opt_params = config.params
cv_model = lgb.LGBMRegressor(**opt_params)

cv_scores = []

for train_idx, val_idx in tcsv.split(X_train):
    X_tr , X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    cv_model.fit(X_tr, y_tr,
                 eval_set=[(X_val, y_val)],
                 eval_metric=['mae'],
                 callbacks = [lgb.early_stopping(stopping_rounds=30)])

    preds = cv_model.predict(X_val)
    preds_raw = np.expm1(preds)
    y_val_raw = np.expm1(y_val)

    fold_mae = mean_absolute_error(y_val_raw, preds_raw)
    cv_scores.append(fold_mae)

print(cv_scores)
print(f'Mean CV scores:{np.mean(cv_scores)}')

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[26]	valid_0's l1: 0.989939
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l1: 0.881668
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l1: 0.794265
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l1: 0.871718
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l1: 0.762249
[63.21395763638403, 42.93723745085742, 28.395278247524857, 23.853077407551616, 15.002058996018857]
Mean CV scores:34.68032194766735


In [6]:
preds_baseline = cv_model.predict(X_val)
mae_baseline = mean_absolute_error(np.expm1(y_val), np.expm1(preds_baseline))

X_val_shuffled = X_val.copy()
X_val_shuffled['StockCode'] = X_val['StockCode'].sample(frac=1, random_state=42).values

preds_shuffled = cv_model.predict(X_val_shuffled)
mae_shuffled = mean_absolute_error(np.expm1(y_val), np.expm1(preds_shuffled))

dependency = mae_shuffled / mae_baseline

print(f"Original MAE: {mae_baseline:.2f}")
print(f"Shuffled MAE: {mae_shuffled:.2f}")
print(f"The model is {dependency:.1f}x worse without the correct StockCode.")

Original MAE: 15.00
Shuffled MAE: 23.41
The model is 1.6x worse without the correct StockCode.


In [20]:
def objective(trial):
    tscv = TimeSeriesSplit(n_splits=5)
    param = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': 1000,

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'cat_smooth': trial.suggest_float('cat_smooth', 10.0, 50.0)
    }

    cv_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMRegressor(**param)


        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )

        preds = model.predict(X_val)

        preds_raw = np.expm1(preds)
        y_val_raw = np.expm1(y_val)

        fold_mae = mean_absolute_error(y_val_raw, preds_raw)

        cv_scores.append(fold_mae)

    return np.mean(cv_scores)

In [ ]:
print("Starting Optuna optimization...")
study = optuna.create_study(direction='minimize')

study.optimize(objective, n_trials=25)

print("========================================")
print(f"Best Validation MAE: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")
print("========================================")

In [6]:
def calculate_naive_cv(X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    naive_scores = []

    for train_idx, val_idx in tscv.split(X):
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        last_value = y_tr.iloc[-1]
        naive_preds = np.full(shape=len(y_val), fill_value=last_value)

        naive_mae = mean_absolute_error(np.expm1(y_val), np.expm1(naive_preds))
        naive_scores.append(naive_mae)

    return np.mean(naive_scores)

In [7]:
naive_cv_mae = calculate_naive_cv(X_train, y_train)
model_cv_mae = study.best_value

improvement = (1 - (model_cv_mae / naive_cv_mae)) * 100

print(f"Naive Baseline CV-MAE:  {naive_cv_mae:.4f}")
print(f"Optuna Best CV-MAE:     {model_cv_mae:.4f}")
print(f"Model Improvement:      {improvement:.2f}%")

Naive Baseline CV-MAE:  52.7073
Optuna Best CV-MAE:     28.9318
Model Improvement:      45.11%


In [9]:
print("Training Final Production Model")


final_train_data = lgb.Dataset(X_train, label=y_train)

final_model = lgb.train(
    opt_params,
    final_train_data,
    num_boost_round=150
)



Training Final Production Model


In [13]:
joblib.dump(final_model, MODELS_DIR / 'MarketDemandModel.joblib')

['C:\\Users\\UserB\\Documents\\ML\\Online Retail II UCI\\models\\MarketDemandModel.joblib']

In [10]:
importance_values = final_model.feature_importance(importance_type='gain')
feature_names = final_model.feature_name()

total_gain = sum(importance_values)

In [11]:
importance_dict = dict(zip(feature_names, ((importance_values / total_gain) * 100)))

In [12]:
sorted_impact = dict(sorted(importance_dict.items(), key=lambda item: item[1], reverse=True))
print(sorted_impact)

{'StockCode': np.float64(21.810365577898295), 'EWMA_Target': np.float64(14.864134362365455), 'Quantity': np.float64(12.11145308865292), 'lag_1t': np.float64(5.47439935319201), 'Unique_Customers': np.float64(5.410422870657963), 'Price_Shock': np.float64(5.304952209732831), 'lag_4t': np.float64(5.1721190359013525), 'lag_2t': np.float64(4.890453941503065), 'lag_3t': np.float64(4.613790746094958), 'Revenue': np.float64(4.23766269313957), 'Price': np.float64(3.62446492333146), 'Price_vs_Avg': np.float64(3.3471653812047495), 'Price_x_Trend': np.float64(3.0596905710145887), 'Month_cos': np.float64(2.6932552571054855), 'Month_sin': np.float64(2.115067128894709), 'ticks_since_last_sale': np.float64(0.619809453211084), 'Holiday_flag': np.float64(0.29348794884677193), 'Price_fall_flag': np.float64(0.18375908284924378), 'Price_rise_flag': np.float64(0.17354637440348197)}
